# Phase 1B: Leakage-safe pre-game features (2025)

**목적:** Phase 1A에서 확인된 정규시즌 2,430경기를 기준으로 *경기 이전 날짜*까지의 팀 성적과 투수 경기별 기록을 재구성합니다. **이번 노트북에서는 모델을 학습하지 않습니다.**

1. 1A의 `mlb_2025_regular_season_verified_2430.csv`를 업로드합니다.
2. MLB Stats API에서 종료 경기 스코어·선발 예정 투수 ID를 확인합니다.
3. 팀 기록은 **동일한 날짜의 다른 경기까지 모두 처리한 뒤** 다음 날짜에만 반영합니다.
4. 투수 ERA/WHIP/K9는 공식 게임로그에서 **해당 경기일 이전** 기록만 합산합니다. 등판 전 기록이 없으면 명시적인 중립 대체값을 넣고 표시합니다.
5. 결과를 `phase1b_output.zip`으로 저장합니다.

**주의:** 이 노트북은 검증을 위해 보수적인 날짜 기준을 사용합니다. 동일 날짜 더블헤더 2차전의 1차전 결과도 포함하지 않으며, 이는 정보 누수 방지에 유리하지만 실제 2차전 직전 이용 가능한 정보보다 적을 수 있습니다. API의 경기별 선발 예정 투수 표시는 실제 선발과 다를 수 있으므로 커버리지 보고서를 확인해야 합니다. 소스 데이터가 확인되지 않은 경기에는 결과를 임의로 채우지 않습니다.


In [ ]:
!pip -q install pandas numpy requests tqdm
import pandas as pd, numpy as np, requests, time, json, zipfile, io
from pathlib import Path
from collections import defaultdict
from tqdm.auto import tqdm
from google.colab import files
OUT=Path('/content/phase1b_output'); OUT.mkdir(exist_ok=True)
BASE='https://statsapi.mlb.com/api/v1'
s=requests.Session();s.headers.update({'User-Agent':'Academic research sample audit; contact researcher'})
def get(path,params=None,retries=4):
    url=BASE+path
    for k in range(retries):
        try:
            r=s.get(url,params=params,timeout=35)
            if r.status_code in (429,500,502,503,504): raise RuntimeError(f'HTTP {r.status_code}')
            r.raise_for_status();return r.json()
        except Exception as e:
            if k==retries-1: raise
            time.sleep(1.5*(k+1))
print('라이브러리 준비 완료')

In [ ]:
print('Phase 1A 검증 CSV를 업로드하세요')
uploaded=files.upload()
files_csv=[k for k in uploaded if k.lower().endswith('.csv')]
assert len(files_csv)==1, 'CSV 한 개만 업로드하세요.'
verified=pd.read_csv(io.BytesIO(uploaded[files_csv[0]]),dtype={'gamePk':int,'home_id':int,'away_id':int})
assert len(verified)==2430 and verified.gamePk.nunique()==2430, '2,430 고유 경기 검증 파일이 아닙니다.'
assert set(verified.gameType)=={'R'} and set(verified.codedGameState)=={'F'}
verified['date']=pd.to_datetime(verified['date']).dt.strftime('%Y-%m-%d')
IDSET=set(verified.gamePk.astype(int)); print('검증 대상',len(IDSET),'경기')

## A. 경기 정보 조회 및 원본 표본과 대조
월 단위로 조회하므로 API 호출이 비교적 적습니다. 원래 목록의 모든 gamePk가 응답에 존재해야 합니다. 각 경기의 팀 ID, 득점, 승패, 선발 예정 투수 ID를 보존합니다.

In [ ]:
months=pd.period_range('2025-03','2025-10',freq='M')
all_raw=[]
for m in tqdm(months,desc='schedule months'):
    p={'sportId':1,'startDate':str(m.start_time.date()),'endDate':str(m.end_time.date()),'hydrate':'probablePitcher'}
    result=get('/schedule',p)
    for d in result.get('dates',[]):
        for g in d.get('games',[]): all_raw.append(g)
    time.sleep(0.15)
byid=defaultdict(list)
for g in all_raw: byid[int(g['gamePk'])].append(g)
missing=sorted(IDSET-set(byid))
assert not missing, f'공식 일정 응답에 {len(missing)}경기가 없습니다. 예: {missing[:10]}'
rows=[]; ambiguities=[]
for rec in verified.itertuples(index=False):
    gs=byid[int(rec.gamePk)]
    gs=sorted(gs,key=lambda g:(g.get('status',{}).get('codedGameState')=='F', bool(g.get('teams',{}).get('home',{}).get('score') is not None)),reverse=True)
    g=gs[0]; h=g['teams']['home'];a=g['teams']['away']
    if (int(h['team']['id']),int(a['team']['id']))!=(int(rec.home_id),int(rec.away_id)):
        raise ValueError(f'팀 ID 불일치: {rec.gamePk}')
    hd=h.get('score');ad=a.get('score')
    if hd is None or ad is None: ambiguities.append((rec.gamePk,'score missing'))
    hp=h.get('probablePitcher') or {};ap=a.get('probablePitcher') or {}
    rows.append({'gamePk':int(rec.gamePk),'date':rec.date,'home_id':int(rec.home_id),'away_id':int(rec.away_id),
      'home_score':hd,'away_score':ad,'home_pitcher_id':hp.get('id'),'away_pitcher_id':ap.get('id'),
      'home_pitcher_name':hp.get('fullName'),'away_pitcher_name':ap.get('fullName'),
      'status':g.get('status',{}).get('detailedState')})
games=pd.DataFrame(rows).sort_values(['date','gamePk']).reset_index(drop=True)
if ambiguities: raise ValueError(f'누락 스코어가 있어 중단: {ambiguities[:10]}')
if (games.home_score==games.away_score).any(): raise ValueError('승패가 없는 동점 기록이 존재합니다.')
games['target']=(games.home_score>games.away_score).astype(int)
games.to_csv(OUT/'games_2430_with_scores.csv',index=False)
print('확인 경기',len(games),'\n선발 예정 투수 ID 결측:',games[['home_pitcher_id','away_pitcher_id']].isna().sum().to_dict())
display(games.head(3))

## B. 팀 누적 기록 — 미래 정보 제외
동일 경기일의 모든 경기 입력변수를 **해당 날짜 시작 전 기록**으로 계산합니다. 10경기 미만의 누적 승률은 원문대로 0.5로 대체하며, 대체 여부를 추적합니다. 득점/실점 평균은 이전 경기 기록만 사용합니다.

In [ ]:
t=defaultdict(lambda:{'gp':0,'w':0,'rs':0,'ra':0,'results':[]})
team_rows=[]
for d,day in games.groupby('date',sort=True):
    # 단계 1: 하루의 모든 경기 특성을 당일 업데이트 전에 계산
    for g in day.itertuples(index=False):
        r={'gamePk':g.gamePk,'date':d,'target':g.target,'home_id':g.home_id,'away_id':g.away_id,
           'home_pitcher_id':g.home_pitcher_id,'away_pitcher_id':g.away_pitcher_id}
        for side,tid in [('home',g.home_id),('away',g.away_id)]:
            v=t[tid];gp=v['gp']
            r[f'{side}_prior_games']=gp
            r[f'{side}_win_pct']=v['w']/gp if gp>=10 else .5
            r[f'{side}_winpct_imputed']=(gp<10)
            r[f'{side}_runs_per_game']=v['rs']/gp if gp else 4.5
            r[f'{side}_ra_per_game']=v['ra']/gp if gp else 4.5
            r[f'{side}_run_diff']=(v['rs']-v['ra'])/gp if gp else 0.
            last=v['results'][-10:];r[f'{side}_recent_form']=float(np.mean(last)) if last else .5
        team_rows.append(r)
    # 단계 2: 그 날짜 모든 경기 결과를 다음 날짜에 반영
    for g in day.itertuples(index=False):
        for side,tid,rs,ra,w in [('home',g.home_id,g.home_score,g.away_score,g.target),('away',g.away_id,g.away_score,g.home_score,1-g.target)]:
            v=t[tid];v['gp']+=1;v['w']+=int(w);v['rs']+=int(rs);v['ra']+=int(ra);v['results'].append(int(w))
team=pd.DataFrame(team_rows)
assert len(team)==2430 and team.gamePk.nunique()==2430
assert all(v['gp']==162 for v in t.values()),'팀별 162경기 여부를 확인하세요.'
team.to_csv(OUT/'team_features_asof_prior_date.csv',index=False)
print('팀 30개 모두 162경기 확인 / 10경기 미만 승률 대체 게임 수:',int((team.home_winpct_imputed|team.away_winpct_imputed).sum()))
display(team.head(3))

## C. 선발투수 기록 — 경기일 이전 Game Log
선발 예정 투수 ID를 사용하여 투수별 2025 정규시즌 gameLog를 조회합니다. 각 경기 **이전 날짜**의 투구이닝·자책점·볼넷·피안타·탈삼진을 누적해 ERA/WHIP/K9를 계산합니다. 공백/미등판/조회 실패는 개별 사유와 함께 별도 파일에 남깁니다.

**중립 대체값:** 미등판자 또는 선발 미확정 시 ERA=4.30, WHIP=1.30, K9=8.50을 사용합니다. 이 값들은 검증된 리그 평균이 아닌 임시 사전 지정 값이므로, 최종 모델 학습 전에 학습셋 기반 평균으로 대체할지 결정해야 합니다. **진단용 값임을 명시합니다.**

In [ ]:
def ip_to_outs(v):
    # MLB IP 표기 5.2 = 5이닝 2아웃; 소수점 계산 금지
    st=str(v or '0.0'); a,b=(st.split('.')+['0'])[:2]
    if b not in ('0','1','2'): raise ValueError(f'IP 표기 이상: {v}')
    return int(a)*3+int(b)
assert ip_to_outs('5.2')==17 and ip_to_outs('6.1')==19
ids=pd.to_numeric(pd.concat([games.home_pitcher_id,games.away_pitcher_id]),errors='coerce').dropna().astype(int).unique()
print('조회할 고유 선발 예정 투수:',len(ids))
logs=[]; problems=[]
for pid in tqdm(ids,desc='pitcher logs'):
    try:
        data=get(f'/people/{int(pid)}/stats',{'stats':'gameLog','group':'pitching','season':2025,'gameType':'R'})
        blocks=data.get('stats',[])
        splits=[x for b in blocks for x in b.get('splits',[])]
        for z in splits:
            dt=z.get('date') or z.get('gameDate')
            if not dt: continue
            st=z.get('stat',{})
            logs.append({'pitcher_id':int(pid),'date':str(dt)[:10],
                         'gamePk':(z.get('game') or {}).get('gamePk'),
                         'outs':ip_to_outs(st.get('inningsPitched','0.0')),
                         'er':int(st.get('earnedRuns',0) or 0),
                         'h':int(st.get('hits',0) or 0),
                         'bb':int(st.get('baseOnBalls',0) or 0),
                         'so':int(st.get('strikeOuts',0) or 0)})
    except Exception as e:
        problems.append({'pitcher_id':int(pid),'error':str(e)})
    time.sleep(.05)
logs=pd.DataFrame(logs,columns=['pitcher_id','date','gamePk','outs','er','h','bb','so'])
if not logs.empty:
    logs=logs.drop_duplicates(['pitcher_id','gamePk'],keep='first') if logs.gamePk.notna().all() else logs.drop_duplicates(['pitcher_id','date','gamePk'],keep='first')
    logs.sort_values(['pitcher_id','date'],inplace=True)
logs.to_csv(OUT/'pitcher_gamelogs_2025.csv',index=False)
pd.DataFrame(problems,columns=['pitcher_id','error']).to_csv(OUT/'pitcher_api_errors.csv',index=False)
print('게임로그',len(logs),'행 / API 오류',len(problems))
if len(problems):print('API 오류 예시',problems[:5])
if logs.empty: raise RuntimeError('투수 게임로그를 전혀 수집하지 못했습니다. ZIP 대신 오류 메시지를 전달해 주세요.')
display(logs.head(5))

In [ ]:
# 날짜 이전만 사용하는 누적 기록 (오늘 기록은 포함되지 않음)
neutral={'era':4.30,'whip':1.30,'k9':8.50}
by_pid={int(k):v.sort_values('date') for k,v in logs.groupby('pitcher_id')}
lookup={}
for g in games.itertuples(index=False):
    for side in ('home','away'):
        pid=getattr(g,f'{side}_pitcher_id')
        if pd.isna(pid):
            lookup[(g.gamePk,side)]=dict(neutral,source='missing_pitcher_id',prior_outs=0);continue
        prior=by_pid.get(int(pid))
        if prior is None:
            lookup[(g.gamePk,side)]=dict(neutral,source='no_gamelog',prior_outs=0);continue
        v=prior.loc[prior.date<g.date,['outs','er','h','bb','so']].sum()
        outs=int(v['outs'])
        if outs==0:
            lookup[(g.gamePk,side)]=dict(neutral,source='no_prior_outs',prior_outs=0);continue
        ip=outs/3
        lookup[(g.gamePk,side)]={'era':9*v['er']/ip,'whip':(v['h']+v['bb'])/ip,'k9':9*v['so']/ip,
                                  'source':'prior_game_logs','prior_outs':outs}
feat=team.copy()
for side in ('home','away'):
    for key in ['era','whip','k9','source','prior_outs']:
        feat[f'{side}_sp_{key}']=[lookup[(int(gid),side)][key] for gid in feat.gamePk]
feat['win_pct_diff']=feat.home_win_pct-feat.away_win_pct
feat['run_diff_diff']=feat.home_run_diff-feat.away_run_diff
feat['form_diff']=feat.home_recent_form-feat.away_recent_form
feat['sp_era_diff']=feat.away_sp_era-feat.home_sp_era # 기존 코드의 방향 유지
feat['sp_whip_diff']=feat.away_sp_whip-feat.home_sp_whip
feat['sp_k9_diff']=feat.home_sp_k9-feat.away_sp_k9
# home_advantage는 모든 표본에서 1인 상수이므로 분석 입력에서 제외 (22개)
features=['home_win_pct','away_win_pct','win_pct_diff','home_run_diff','away_run_diff','run_diff_diff',
          'home_runs_per_game','away_runs_per_game','home_ra_per_game','away_ra_per_game',
          'home_recent_form','away_recent_form','form_diff',
          'home_sp_era','away_sp_era','sp_era_diff','home_sp_whip','away_sp_whip','sp_whip_diff',
          'home_sp_k9','away_sp_k9','sp_k9_diff']
assert len(features)==22 and feat[features].notna().all().all()
feat.to_csv(OUT/'pregame_asof_prior_date_DIAGNOSTIC.csv',index=False)
print('2,430경기 특성 계산 완료 (현재는 진단용, 재학습 전 대체값 재검토 필요)')
print('선발 기록 출처:')
for side in ('home','away'):print(side,feat[f'{side}_sp_source'].value_counts().to_dict())
display(feat[['gamePk','date','home_prior_games','away_prior_games','home_sp_source','away_sp_source']].head(8))

## D. 진단 결과 다운로드
진단 결과(`audit_report.json`, `pregame_asof_prior_date_DIAGNOSTIC.csv`, `pitcher_api_errors.csv` 포함)를 ZIP 파일로 저장합니다. 모델 학습은 다음 단계에서 수행합니다.

In [ ]:
report={'season':2025,'regular_games':len(games),'unique_games':int(games.gamePk.nunique()),
        'unique_pitcher_ids':len(ids),'gamelog_rows':len(logs),'pitcher_api_failures':len(problems),
        'missing_pitcher_ids':{side:int(games[f'{side}_pitcher_id'].isna().sum()) for side in ('home','away')},
        'pitcher_feature_sources':{side:feat[f'{side}_sp_source'].value_counts().to_dict() for side in ('home','away')},
        'early_win_pct_imputed_games':int((feat.home_winpct_imputed|feat.away_winpct_imputed).sum()),
        'date_policy':'strictly earlier dates; all same-date games excluded from prior stats',
        'pitcher_defaults':'diagnostic only: ERA 4.30, WHIP 1.30, K9 8.50',
        'feature_count':len(features),'feature_names':features,
        'limitations':['probablePitcher may differ from actual starter','diagnostic neutral defaults not fitted on train only','model not retrained']}
(OUT/'audit_report.json').write_text(json.dumps(report,ensure_ascii=False,indent=2))
zipname='/content/phase1b_output.zip'
with zipfile.ZipFile(zipname,'w',compression=zipfile.ZIP_DEFLATED) as z:
    for p in OUT.iterdir():z.write(p,arcname=p.name)
print(json.dumps({k:v for k,v in report.items() if k!='feature_names'},ensure_ascii=False,indent=2))
print('\n완료:',zipname)
files.download(zipname)